# 03 — DCGAN on MNIST (TensorFlow/Keras)

TensorFlow-only implementation using Keras subclassing and `GradientTape`. MNIST is downloaded through Keras.

## 1. Environment and data

In [ ]:
import tensorflow as tf, matplotlib.pyplot as plt
print('TensorFlow',tf.__version__)
(x,_),_=tf.keras.datasets.mnist.load_data()
x=(x.astype('float32')/127.5)-1.; x=x[...,None]
train=tf.data.Dataset.from_tensor_slices(x).shuffle(60000).batch(128,drop_remainder=True).prefetch(tf.data.AUTOTUNE)

## 2. Generator and discriminator

In [ ]:
from tensorflow import keras
from tensorflow.keras import layers
LATENT=100
G=keras.Sequential([layers.Input((LATENT,)),layers.Dense(7*7*256,use_bias=False),layers.BatchNormalization(),layers.ReLU(),layers.Reshape((7,7,256)),layers.Conv2DTranspose(128,5,2,padding='same',use_bias=False),layers.BatchNormalization(),layers.ReLU(),layers.Conv2DTranspose(64,5,2,padding='same',use_bias=False),layers.BatchNormalization(),layers.ReLU(),layers.Conv2DTranspose(1,5,1,padding='same',activation='tanh')])
D=keras.Sequential([layers.Input((28,28,1)),layers.Conv2D(64,5,2,padding='same'),layers.LeakyReLU(.2),layers.Dropout(.3),layers.Conv2D(128,5,2,padding='same'),layers.LeakyReLU(.2),layers.Dropout(.3),layers.Flatten(),layers.Dense(1)])

## 3. Losses and train step

Binary cross-entropy with logits: discriminator learns real=1/fake=0; generator learns to make fake predictions close to 1.

In [ ]:
bce=keras.losses.BinaryCrossentropy(from_logits=True)
g_opt=keras.optimizers.Adam(2e-4,beta_1=.5); d_opt=keras.optimizers.Adam(2e-4,beta_1=.5)
@tf.function
def train_step(real):
 n=tf.shape(real)[0]; z=tf.random.normal((n,LATENT))
 with tf.GradientTape() as gt, tf.GradientTape() as dt:
  fake=G(z,training=True); real_logits=D(real,training=True); fake_logits=D(fake,training=True)
  d_loss=bce(tf.ones_like(real_logits),real_logits)+bce(tf.zeros_like(fake_logits),fake_logits)
  g_loss=bce(tf.ones_like(fake_logits),fake_logits)
 gd=gt.gradient(g_loss,G.trainable_variables); dd=dt.gradient(d_loss,D.trainable_variables)
 g_opt.apply_gradients(zip(gd,G.trainable_variables)); d_opt.apply_gradients(zip(dd,D.trainable_variables))
 return g_loss,d_loss

## 4. Train and inspect fixed-noise samples

In [ ]:
fixed_z=tf.random.normal((16,LATENT))
for epoch in range(10):
 gl=[]; dl=[]
 for real in train:
  g_loss,d_loss=train_step(real); gl.append(g_loss); dl.append(d_loss)
 print(epoch+1,float(tf.reduce_mean(gl)),float(tf.reduce_mean(dl)))
 samples=G(fixed_z,training=False)
 fig,ax=plt.subplots(4,4,figsize=(5,5))
 for i,a in enumerate(ax.flat): a.imshow((samples[i,:,:,0]+1)/2,cmap='gray'); a.axis('off')
 plt.show()

## GAN stability tips

- Track generated samples, not only scalar losses.
- Use fixed latent vectors to compare epochs.
- Mode collapse means low sample diversity.
- Save checkpoints and random seeds.
- Start with a small epoch count for smoke testing; GANs may require careful tuning.

## Exercises
Try latent dimensions 64/100/128, label smoothing for real labels, and compare diversity across seeds.